# ViPRA baseline — LoRA FP16, Qwen2.5-Math-7B-Instruct

Backbone **7,62B**, dưới giới hạn 8B của cuộc thi. LoRA FP16, không lượng tử hóa.
Chọn **T4 x2**, bật Internet, Add Input chứa hai CSV. Clone repo về
`/kaggle/working/VLSP` trước khi chạy notebook này:

```bash
git clone --branch Nhotest --single-branch https://github.com/NgocNhoNguyen2605n/VLSP.git /kaggle/working/VLSP
```

Pipeline giữ nguyên văn bản CSV; chỉ bỏ train thiếu answer và ghi ID.
Chạy các cell theo thứ tự. Seed 42, greedy, không có thanh phần trăm.


In [ ]:
import os  # Thiết lập môi trường cho các tiến trình train/evaluate.
import subprocess  # Chạy Python riêng để trả lại VRAM sau mỗi tác vụ.
import sys  # Dùng đúng Python của notebook Kaggle.
from pathlib import Path  # Đường dẫn repo, Kaggle Input và kết quả.

REPO = Path("/kaggle/working/VLSP")
DATA_ROOT = "/kaggle/input"
RUN_DIR = "/kaggle/working/vipra_qwen25_math_7b_seed42"  # Chọn thư mục mới cho mỗi lần train.
assert (REPO / "requirements.txt").is_file(), "Hãy clone repo vào /kaggle/working/VLSP trước."

os.environ.update(PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8",
                  TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1",
                  HF_HOME="/tmp/vipra_hf_cache")  # Không lưu backbone vào output notebook.

def run(module, *options):
    "Chạy tác vụ trong repo, dừng notebook nếu có lỗi."
    subprocess.run([sys.executable, "-m", module, *options], cwd=REPO, check=True)

# Gỡ torchao cũ của Kaggle: PEFT kiểm tra cả khi chỉ dùng LoRA FP16.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                "torchao"], check=True)

# Cài các phiên bản cố định; giữ PyTorch CUDA sẵn có của Kaggle.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO / "requirements.txt")], check=True)


## 1. Kiểm tra dữ liệu và template

Tự tìm CSV trong Input. Nếu có nhiều bản cùng tên, thu hẹp `DATA_ROOT`
hoặc truyền `--train-file` và `--dev-file` cụ thể vào cả check/train.
Không tải trọng số backbone trong bước này.


In [ ]:
# Kiểm tra các mẫu và mask assistant; mẫu dài sẽ báo lỗi thay vì bị cắt.
run("baseline.check", "--data-root", DATA_ROOT)


## 2. Huấn luyện LoRA

Một tiến trình chia backbone FP16 giữa hai T4. Không dùng `torchrun`.
Micro-batch 1, tích lũy gradient 8 (effective batch 8), max_length 2048.
Trọng số FP16 khoảng 14,19 GiB tổng; chưa gồm activation/adapter/optimizer.
Mỗi epoch lưu một adapter; adapter cuối dùng để chấm dev. Nếu thư mục
run đã có dữ liệu, đổi `RUN_DIR` để giữ kết quả cũ.

Log ghi số mẫu sau batch đầu, sau khoảng 60 giây khi batch hoàn thành
và ở các mốc cập nhật; không dùng thanh phần trăm.

FP16 bắt đầu với scale 128, tự giảm khi overflow; log ghi số bước bỏ.
Dừng nếu overflow 8 bước liên tiếp; không chấm dev khi train chưa hoàn tất.

Attention dùng SDPA math với trung gian FP32 để tránh tràn FP16 trước softmax.
Không cần FlashAttention hoặc BF16 trên T4. Nếu loss vẫn NaN/Inf, log báo
projection/norm đầu tiên bị lỗi và run.json ghi trạng thái failed.
Sau khi run thất bại, dùng RUN_DIR mới để không ghi đè dữ liệu chẩn đoán.


In [ ]:
# Hyperparameter ở configs/baseline.json; chỉ adapter LoRA nhận gradient.
run("baseline.train", "--data-root", DATA_ROOT, "--output-dir", RUN_DIR)


## 3. Đánh giá dev

P1 **ước lượng**: đáp án và đơn vị đúng ở mọi phần, explanation không rỗng.
Dung sai tạm `rtol=0.01`, `atol=0`, chưa phải evaluator chính thức.
Sinh greedy một lần mỗi câu; lưu cả raw JSON, thời gian và cờ đúng/sai.
Đường PyTorch trên T4 có thể chậm: xem `content_tolerant` để tách chất lượng
đáp án khỏi các câu quá giới hạn 60 giây.


In [ ]:
# Đọc lại cấu hình và dataset hash đã lưu lúc train, không đưa gold vào prompt.
assert (Path(RUN_DIR) / "adapter" / "adapter_config.json").is_file(), "Hãy train thành công trước khi evaluate."
run("baseline.evaluate", "--run-dir", RUN_DIR)


Kết quả: `RUN_DIR/evaluation/metrics.json` và `predictions.jsonl`.
Muốn chấm lại cùng adapter, truyền `--result-dir` trống khác để không ghi đè.
Seed cố định giúp tái lập; không bảo đảm giống bit giữa môi trường CUDA khác nhau.
